 Cell 1 — Imports   

In [3]:
!pip install imbalanced-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]


Cell 2 — Load raw data 

In [4]:
df = pd.read_csv("../data/raw/framingham.csv")        
print(df.shape)

(4240, 16)


Cell 3 — Median imputation

In [5]:
for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Missing values remaining:", df.isnull().sum().sum())

Missing values remaining: 0


Cell 4 — Train / Val / Test split (70/15/15)

In [6]:
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (2968, 15), Val: (636, 15), Test: (636, 15)


Cell 5 — Feature scaling

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit ONLY on train
X_val_scaled   = scaler.transform(X_val)         # transform only
X_test_scaled  = scaler.transform(X_test)         # transform only

Cell 6 - Apply Smote


In [8]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Original training class distribution:")
print(pd.Series(y_train).value_counts())
print("\nResampled training class distribution:")
print(pd.Series(y_train_resampled).value_counts())

Original training class distribution:
TenYearCHD
0    2517
1     451
Name: count, dtype: int64

Resampled training class distribution:
TenYearCHD
0    2517
1    2517
Name: count, dtype: int64


Cell 6 — Save processed data

In [9]:
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save resampled train (with SMOTE)
pd.DataFrame(X_train_resampled, columns=X.columns)\
  .assign(TenYearCHD=y_train_resampled)\
  .to_csv("../data/processed/train.csv", index=False)

# Save val and test (no SMOTE — real data only)
pd.DataFrame(X_val_scaled, columns=X.columns)\
  .assign(TenYearCHD=y_val.values)\
  .to_csv("../data/processed/val.csv", index=False)

pd.DataFrame(X_test_scaled, columns=X.columns)\
  .assign(TenYearCHD=y_test.values)\
  .to_csv("../data/processed/test.csv", index=False)

# Save scaler for teammates
joblib.dump(scaler, "../models/scaler.pkl")

print("Saved train (SMOTE), val, test to data/processed/")
print("Saved scaler to models/scaler.pkl")

Saved train (SMOTE), val, test to data/processed/
Saved scaler to models/scaler.pkl
